# 📊 Notebook 01 — Maximum Likelihood Estimation (MLE)

**Mục tiêu của notebook này:**
- Hiểu MLE bằng trực giác, không chỉ công thức
- Tự tay tính MLE cho Gaussian và Bernoulli
- Vẽ đồ thị để "thấy" log-likelihood surface

---
**Cách dùng:** Chạy từng cell theo thứ tự. Đọc kỹ comment và thử thay đổi các tham số để quan sát.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import sys, os

# Thêm thư mục gốc của repo vào path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.mle import GaussianMLE, BernoulliMLE
from src.utils import make_gaussian, make_bernoulli

# Cài đặt đồ thị cho đẹp
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ Import thành công!')

---
## Phần 1: Trực giác về MLE

> **Câu hỏi cốt lõi:** Nếu bạn quan sát được data này, tham số nào của mô hình khiến data có xác suất xuất hiện **cao nhất**?

Hãy tưởng tượng bạn đo chiều cao của 10 người và được: `[168, 172, 165, 170, 175, 163, 169, 171, 167, 174]`

Bạn giả sử chiều cao ~ N(μ, σ²). MLE hỏi: **μ và σ nào phù hợp nhất với data này?**

In [ ]:
# Data chiều cao (cm)
height_data = [168, 172, 165, 170, 175, 163, 169, 171, 167, 174]

# Thử 3 giá trị μ khác nhau — cái nào "fit" data nhất?
mu_candidates = [160, 169.4, 180]  # thấp, vừa, cao
sigma_fixed = 4.0

x_range = np.linspace(150, 190, 500)
colors = ['#e74c3c', '#2ecc71', '#e67e22']
labels = ['μ=160 (quá thấp)', 'μ=169.4 (MLE)', 'μ=180 (quá cao)']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, mu, color, label in zip(axes, mu_candidates, colors, labels):
    # Vẽ phân phối Gaussian
    ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma_fixed),
            color=color, lw=2.5, label=label)
    
    # Vẽ data points (dấu gạch đỏ)
    for x in height_data:
        ax.axvline(x, color='gray', alpha=0.4, lw=1)
    
    # Tính log-likelihood
    ll = np.sum(stats.norm.logpdf(height_data, mu, sigma_fixed))
    ax.set_title(f'{label}\nlog L = {ll:.1f}', fontsize=10)
    ax.set_xlabel('Chiều cao (cm)')
    ax.set_xlim(148, 192)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.suptitle('Cùng data, 3 giá trị μ khác nhau — log L càng cao càng tốt', 
             y=1.05, fontsize=12, fontweight='bold')
plt.show()

print('👉 μ = 169.4 có log-likelihood cao nhất → đó là MLE!')

---
## Phần 2: Log-Likelihood Surface cho Gaussian

Hãy vẽ log-likelihood như một bề mặt 2D — **đỉnh của bề mặt chính là MLE**.

In [ ]:
# Sinh data từ N(5.0, 2.0)
np.random.seed(42)
TRUE_MU, TRUE_SIGMA = 5.0, 2.0
data = make_gaussian(n=30, mu=TRUE_MU, sigma=TRUE_SIGMA, seed=42)

# Grid các giá trị μ và σ để thử
mu_grid    = np.linspace(2, 8, 100)
sigma_grid = np.linspace(0.5, 4, 100)
MU, SIGMA  = np.meshgrid(mu_grid, sigma_grid)

# Tính log-likelihood tại mỗi điểm (μ, σ)
LL = np.zeros_like(MU)
for i in range(MU.shape[0]):
    for j in range(MU.shape[1]):
        LL[i, j] = np.sum(stats.norm.logpdf(data, MU[i,j], SIGMA[i,j]))

# MLE thực tế
mu_mle    = np.mean(data)
sigma_mle = np.std(data, ddof=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Contour plot ---
ax = axes[0]
cp = ax.contourf(MU, SIGMA, LL, levels=30, cmap='viridis')
plt.colorbar(cp, ax=ax, label='log-likelihood')
ax.plot(mu_mle, sigma_mle, 'r*', ms=18, label=f'MLE: μ={mu_mle:.2f}, σ={sigma_mle:.2f}')
ax.plot(TRUE_MU, TRUE_SIGMA, 'w+', ms=14, mew=2.5, label=f'Thật: μ={TRUE_MU}, σ={TRUE_SIGMA}')
ax.set_xlabel('μ')
ax.set_ylabel('σ')
ax.set_title('Log-likelihood surface (contour)\n★ = MLE,  + = giá trị thật')
ax.legend(fontsize=9)

# --- Slice: log L theo μ (σ cố định ở MLE) ---
ax2 = axes[1]
ll_slice = [np.sum(stats.norm.logpdf(data, m, sigma_mle)) for m in mu_grid]
ax2.plot(mu_grid, ll_slice, 'steelblue', lw=2)
ax2.axvline(mu_mle, color='red', lw=2, linestyle='--', label=f'μ_MLE = {mu_mle:.2f}')
ax2.axvline(TRUE_MU, color='green', lw=1.5, linestyle=':', label=f'μ thật = {TRUE_MU}')
ax2.set_xlabel('μ')
ax2.set_ylabel('log L(μ | data, σ_MLE)')
ax2.set_title('Log-likelihood theo μ\n(đỉnh = MLE)')
ax2.legend()

plt.tight_layout()
plt.show()

---
## Phần 3: GaussianMLE — dùng thư viện src/

Bây giờ dùng class `GaussianMLE` đã viết sẵn và xem kết quả.

In [ ]:
# --- Thử với cỡ mẫu khác nhau ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sample_sizes = [5, 30, 500]

for ax, n in zip(axes, sample_sizes):
    data_n = make_gaussian(n=n, mu=TRUE_MU, sigma=TRUE_SIGMA, seed=n)
    
    est = GaussianMLE().fit(data_n)
    
    # Histogram data
    ax.hist(data_n, bins='auto', density=True, alpha=0.5, 
            color='steelblue', label='Data')
    
    # Đường MLE fit
    x_range = np.linspace(est.mu - 4*est.sigma, est.mu + 4*est.sigma, 300)
    ax.plot(x_range, stats.norm.pdf(x_range, est.mu, est.sigma),
            'r-', lw=2.5, label=f'MLE fit')
    
    # Đường thật
    ax.plot(x_range, stats.norm.pdf(x_range, TRUE_MU, TRUE_SIGMA),
            'g--', lw=1.5, label='Phân phối thật')
    
    error = abs(est.mu - TRUE_MU)
    ax.set_title(f'n = {n}\nμ_MLE={est.mu:.2f}  (sai {error:.2f})', fontsize=10)
    ax.set_xlabel('x')
    if n == 5: ax.legend(fontsize=8)

plt.suptitle('MLE Gaussian — n càng lớn, ước lượng càng chính xác', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('📌 Quan sát: n=5 sai nhiều, n=500 gần như trùng với giá trị thật!')

---
## Phần 4: BernoulliMLE — tung đồng xu

Giả sử đồng xu có xác suất ngửa thật là **p = 0.7**. MLE ước lượng p từ data.

In [ ]:
TRUE_P = 0.7

# Mô phỏng: p_MLE hội tụ về p thật khi n tăng
n_values = range(1, 201)
p_mle_list = []

rng = np.random.default_rng(seed=0)
cumulative_data = rng.binomial(1, TRUE_P, size=200)

for n in n_values:
    subset = cumulative_data[:n]
    p_mle_list.append(np.mean(subset))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Convergence plot ---
ax = axes[0]
ax.plot(n_values, p_mle_list, 'steelblue', lw=1.5, alpha=0.8, label='p_MLE')
ax.axhline(TRUE_P, color='red', lw=2, linestyle='--', label=f'p thật = {TRUE_P}')
ax.fill_between(n_values,
                [TRUE_P - 1/np.sqrt(n) for n in n_values],
                [TRUE_P + 1/np.sqrt(n) for n in n_values],
                alpha=0.15, color='red', label='±1/√n band')
ax.set_xlabel('Số lần tung (n)')
ax.set_ylabel('p_MLE')
ax.set_title('p_MLE hội tụ về p thật khi n tăng')
ax.legend()
ax.set_ylim(0, 1.05)

# --- Log-likelihood curve ---
ax2 = axes[1]
data_50 = cumulative_data[:50]
k = np.sum(data_50)
n = len(data_50)
p_range = np.linspace(0.01, 0.99, 300)
ll_values = k * np.log(p_range) + (n - k) * np.log(1 - p_range)

p_mle_50 = k / n
ax2.plot(p_range, ll_values, 'steelblue', lw=2)
ax2.axvline(p_mle_50, color='red', lw=2, linestyle='--', 
            label=f'p_MLE = {p_mle_50:.2f}')
ax2.axvline(TRUE_P, color='green', lw=1.5, linestyle=':',
            label=f'p thật = {TRUE_P}')
ax2.set_xlabel('p')
ax2.set_ylabel('log L(p | data)')
ax2.set_title(f'Log-likelihood Bernoulli (n=50, k={k})')
ax2.legend()

plt.tight_layout()
plt.show()

# Dùng class BernoulliMLE
est = BernoulliMLE().fit(data_50.tolist())
est.summary()

---
## Phần 5: MLE có thể thất bại khi n nhỏ

Đây là lý do MAP ra đời.

In [ ]:
print('=' * 50)
print('Thí nghiệm: Tung đồng xu TRUE_P=0.7')
print('=' * 50)

for seed in range(5):
    rng = np.random.default_rng(seed)
    small_data = rng.binomial(1, TRUE_P, size=3).tolist()  # Chỉ 3 lần!
    p_mle = np.mean(small_data)
    print(f'  Seed {seed}: data={small_data}  →  p_MLE = {p_mle:.2f}', end='')
    if p_mle == 0.0 or p_mle == 1.0:
        print('  ⚠️  Cực đoan!')
    else:
        print()

print()
print('👉 Với n=3, MLE rất không ổn định — đây là lúc cần MAP!')
print('   Xem notebook 02_map_intro.ipynb để biết cách giải quyết.')

---
## Tóm tắt

| Điều học được | Công thức |
|---|---|
| MLE Gaussian | μ_MLE = mean(data), σ_MLE = std(data, ddof=0) |
| MLE Bernoulli | p_MLE = k/n |
| MLE tìm gì? | argmax log P(data \| θ) |
| Hạn chế | Không ổn định khi n nhỏ |

**Bước tiếp theo:** Làm bài tập trong `exercises/beginner/ex01_mle_gaussian.py` và `ex02_mle_bernoulli.py`